# LP vs PPO: So sánh phương pháp VM Allocation/Rescheduling

Notebook này so sánh hai phương pháp phân bổ và tái lập lịch máy ảo:
1. **Linear Programming (LP)** - Tối ưu hóa chính xác theo từng bước thời gian
2. **Proximal Policy Optimization (PPO)** - Học tăng cường với hai kịch bản:
   - `overload`: Ưu tiên giảm thiểu vi phạm SLA
   - `cost`: Ưu tiên tối ưu chi phí vận hành

## Tiêu chí so sánh
1. **Tổng số máy ảo** được sử dụng
2. **Tỷ lệ sử dụng tài nguyên** (CPU/Memory utilization)
3. **Chi phí thuê vận hành VM**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Cấu hình hiển thị
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

RESULTS_DIR = Path('forecast_result')
print('✓ Libraries loaded')


## 1. Load dữ liệu từ LP và PPO


In [ ]:
# Load LP reactive schedule
lp_df = pd.read_csv(RESULTS_DIR / 'vm_schedule_notebook_reactive.csv')
lp_df['timestamp'] = pd.to_datetime(lp_df['timestamp'])

# Load PPO schedules (both scenarios)
ppo_cost_df = pd.read_csv(RESULTS_DIR / 'ppo_schedule_test_cost.csv')
ppo_overload_df = pd.read_csv(RESULTS_DIR / 'ppo_schedule_test_overload.csv')

print(f'LP (Reactive):      {len(lp_df):,} steps')
print(f'PPO (Cost):         {len(ppo_cost_df):,} steps')
print(f'PPO (Overload):     {len(ppo_overload_df):,} steps')


In [ ]:
# Preview LP data
print('=== LP Schedule Preview ===')
lp_df[['timestamp', 'vm_plan', 'vm_total_count', 'vm_cost_per_hour', 
       'cpu_utilization_pct', 'mem_utilization_pct', 'sla_violation']].head(10)


In [ ]:
# Preview PPO data
print('=== PPO (Cost) Schedule Preview ===')
ppo_cost_df[['timestamp', 'allocation', 'vm_cost_per_hour', 
             'cpu_utilization_pct', 'mem_utilization_pct', 'sla_violation_flag']].head(10)


## 2. Trích xuất & Tính toán Metrics


In [ ]:
def extract_vm_count_from_allocation(allocation_str):
    """Trích xuất tổng số VM từ chuỗi allocation (e.g., 'B2s×2, D2s_v3×1' -> 3)"""
    if pd.isna(allocation_str) or allocation_str == 'Host only':
        return 0
    total = 0
    parts = str(allocation_str).split(',')
    for part in parts:
        part = part.strip()
        if '×' in part:
            try:
                count = int(part.split('×')[1])
                total += count
            except (ValueError, IndexError):
                pass
    return total

# Tính số VM cho PPO (từ cột allocation)
ppo_cost_df['vm_count'] = ppo_cost_df['allocation'].apply(extract_vm_count_from_allocation)
ppo_overload_df['vm_count'] = ppo_overload_df['allocation'].apply(extract_vm_count_from_allocation)

# LP đã có cột vm_total_count
lp_df['vm_count'] = lp_df['vm_total_count']

print('✓ VM counts extracted')


In [ ]:
def compute_summary_metrics(df, name, cost_col='vm_cost_per_hour', 
                            cpu_util_col='cpu_utilization_pct', 
                            mem_util_col='mem_utilization_pct',
                            sla_col='sla_violation'):
    """Tính các metrics tổng hợp cho một phương pháp"""
    
    # Handle different column names
    if sla_col not in df.columns and 'sla_violation_flag' in df.columns:
        sla_col = 'sla_violation_flag'
    
    n_steps = len(df)
    
    # 1. Tổng số VM
    total_vms = df['vm_count'].sum()
    avg_vms = df['vm_count'].mean()
    max_vms = df['vm_count'].max()
    steps_with_vms = (df['vm_count'] > 0).sum()
    pct_steps_with_vms = steps_with_vms / n_steps * 100
    
    # 2. Tỷ lệ sử dụng tài nguyên (chỉ tính khi có VM)
    vm_steps = df[df['vm_count'] > 0]
    if len(vm_steps) > 0:
        mean_cpu_util = vm_steps[cpu_util_col].mean()
        mean_mem_util = vm_steps[mem_util_col].mean()
    else:
        mean_cpu_util = 0
        mean_mem_util = 0
    
    # Tỷ lệ utilization trung bình toàn bộ (kể cả Host only)
    overall_cpu_util = df[cpu_util_col].mean()
    overall_mem_util = df[mem_util_col].mean()
    
    # 3. Chi phí vận hành
    total_cost = df[cost_col].sum()
    avg_cost_per_step = df[cost_col].mean()
    # Chi phí quy đổi ra đơn vị giờ (mỗi step = 30s = 1/120 giờ)
    # Nếu cost_per_hour thì cost thực tế mỗi step = cost_per_hour * (30/3600) = cost_per_hour / 120
    actual_cost = total_cost / 120  # Tổng chi phí thực tế
    
    # SLA violations
    sla_violations = df[sla_col].sum()
    sla_violation_rate = df[sla_col].mean() * 100
    
    return {
        'Method': name,
        'Total Steps': n_steps,
        # VM metrics
        'Total VMs Used': total_vms,
        'Avg VMs/Step': round(avg_vms, 3),
        'Max VMs': max_vms,
        'Steps with VMs': steps_with_vms,
        '% Steps with VMs': round(pct_steps_with_vms, 2),
        # Utilization metrics
        'Mean CPU Util (VM steps) %': round(mean_cpu_util, 2),
        'Mean Mem Util (VM steps) %': round(mean_mem_util, 2),
        'Overall CPU Util %': round(overall_cpu_util, 2),
        'Overall Mem Util %': round(overall_mem_util, 2),
        # Cost metrics
        'Total Cost ($/hour sum)': round(total_cost, 4),
        'Actual Total Cost ($)': round(actual_cost, 4),
        'Avg Cost/Step ($/h)': round(avg_cost_per_step, 6),
        # SLA
        'SLA Violations': int(sla_violations),
        'SLA Violation Rate %': round(sla_violation_rate, 2),
    }

# Tính metrics cho từng phương pháp
lp_metrics = compute_summary_metrics(lp_df, 'LP (Reactive)')
ppo_cost_metrics = compute_summary_metrics(ppo_cost_df, 'PPO (Cost)')
ppo_overload_metrics = compute_summary_metrics(ppo_overload_df, 'PPO (Overload)')

print('✓ Summary metrics computed')


## 3. Bảng So sánh Tổng hợp


In [ ]:
# Tạo DataFrame so sánh
comparison_df = pd.DataFrame([lp_metrics, ppo_cost_metrics, ppo_overload_metrics])
comparison_df = comparison_df.set_index('Method')

# Transpose để dễ đọc
comparison_df.T


In [ ]:
# Bảng so sánh 3 tiêu chí chính
main_metrics = ['Total VMs Used', 'Avg VMs/Step', 'Mean CPU Util (VM steps) %', 
                'Mean Mem Util (VM steps) %', 'Actual Total Cost ($)', 'SLA Violations']

print('\n' + '='*70)
print('           SO SÁNH 3 TIÊU CHÍ CHÍNH: LP vs PPO')
print('='*70)
comparison_df[main_metrics].T


## 4. Visualizations


In [ ]:
# Color palette
colors = {
    'LP (Reactive)': '#2ecc71',      # Green
    'PPO (Cost)': '#3498db',         # Blue
    'PPO (Overload)': '#e74c3c'      # Red
}

methods = ['LP (Reactive)', 'PPO (Cost)', 'PPO (Overload)']
bar_colors = [colors[m] for m in methods]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Tổng số VM sử dụng
ax1 = axes[0]
vm_totals = [lp_metrics['Total VMs Used'], 
             ppo_cost_metrics['Total VMs Used'], 
             ppo_overload_metrics['Total VMs Used']]
bars1 = ax1.bar(methods, vm_totals, color=bar_colors, edgecolor='white', linewidth=1.5)
ax1.set_title('1. Tổng số VM đã sử dụng', fontweight='bold', pad=15)
ax1.set_ylabel('Số lượng VM')
ax1.set_ylim(0, max(vm_totals) * 1.15 if max(vm_totals) > 0 else 1)
for bar, val in zip(bars1, vm_totals):
    ax1.annotate(f'{val:,}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.tick_params(axis='x', rotation=15)

# 2. Tỷ lệ sử dụng tài nguyên (CPU)
ax2 = axes[1]
cpu_utils = [lp_metrics['Mean CPU Util (VM steps) %'], 
             ppo_cost_metrics['Mean CPU Util (VM steps) %'], 
             ppo_overload_metrics['Mean CPU Util (VM steps) %']]
bars2 = ax2.bar(methods, cpu_utils, color=bar_colors, edgecolor='white', linewidth=1.5)
ax2.set_title('2. Tỷ lệ sử dụng CPU trung bình\n(khi có VM)', fontweight='bold', pad=15)
ax2.set_ylabel('CPU Utilization (%)')
ax2.set_ylim(0, 100)
ax2.axhline(y=70, color='orange', linestyle='--', alpha=0.7, label='Ngưỡng tốt (70%)')
for bar, val in zip(bars2, cpu_utils):
    ax2.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.legend(loc='upper right')
ax2.tick_params(axis='x', rotation=15)

# 3. Chi phí vận hành
ax3 = axes[2]
costs = [lp_metrics['Actual Total Cost ($)'], 
         ppo_cost_metrics['Actual Total Cost ($)'], 
         ppo_overload_metrics['Actual Total Cost ($)']]
bars3 = ax3.bar(methods, costs, color=bar_colors, edgecolor='white', linewidth=1.5)
ax3.set_title('3. Chi phí vận hành VM thực tế', fontweight='bold', pad=15)
ax3.set_ylabel('Chi phí ($)')
ax3.set_ylim(0, max(costs) * 1.15 if max(costs) > 0 else 1)
for bar, val in zip(bars3, costs):
    ax3.annotate(f'${val:.2f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
ax3.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('forecast_result/lp_vs_ppo_comparison.png', dpi=150, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.show()
print('✓ Chart saved to forecast_result/lp_vs_ppo_comparison.png')


In [ ]:
# Chi tiết hơn: So sánh VM count và Cost theo thời gian (rolling average)
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Lấy số step chung nhỏ nhất để so sánh
min_steps = min(len(lp_df), len(ppo_cost_df), len(ppo_overload_df))
window = 60  # Rolling window 30 phút (60 steps × 30s)

# Chuẩn bị dữ liệu (cắt về cùng độ dài)
lp_subset = lp_df.iloc[:min_steps].copy()
ppo_cost_subset = ppo_cost_df.iloc[:min_steps].copy()
ppo_overload_subset = ppo_overload_df.iloc[:min_steps].copy()

# Tạo index thời gian từ 0
x_steps = np.arange(min_steps)
x_hours = x_steps * 30 / 3600  # Chuyển sang giờ

# Plot 1: VM Count (rolling)
ax1 = axes[0]
ax1.plot(x_hours, lp_subset['vm_count'].rolling(window, min_periods=1).mean(), 
         label='LP (Reactive)', color=colors['LP (Reactive)'], linewidth=2, alpha=0.9)
ax1.plot(x_hours, ppo_cost_subset['vm_count'].rolling(window, min_periods=1).mean(), 
         label='PPO (Cost)', color=colors['PPO (Cost)'], linewidth=2, alpha=0.9)
ax1.plot(x_hours, ppo_overload_subset['vm_count'].rolling(window, min_periods=1).mean(), 
         label='PPO (Overload)', color=colors['PPO (Overload)'], linewidth=2, alpha=0.9)
ax1.set_xlabel('Thời gian (giờ)')
ax1.set_ylabel('Số VM (rolling avg 30 phút)')
ax1.set_title('Số lượng VM theo thời gian', fontweight='bold', fontsize=13)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Plot 2: Cost (rolling)
ax2 = axes[1]
ax2.plot(x_hours, lp_subset['vm_cost_per_hour'].rolling(window, min_periods=1).mean(), 
         label='LP (Reactive)', color=colors['LP (Reactive)'], linewidth=2, alpha=0.9)
ax2.plot(x_hours, ppo_cost_subset['vm_cost_per_hour'].rolling(window, min_periods=1).mean(), 
         label='PPO (Cost)', color=colors['PPO (Cost)'], linewidth=2, alpha=0.9)
ax2.plot(x_hours, ppo_overload_subset['vm_cost_per_hour'].rolling(window, min_periods=1).mean(), 
         label='PPO (Overload)', color=colors['PPO (Overload)'], linewidth=2, alpha=0.9)
ax2.set_xlabel('Thời gian (giờ)')
ax2.set_ylabel('Chi phí ($/h, rolling avg 30 phút)')
ax2.set_title('Chi phí VM theo thời gian', fontweight='bold', fontsize=13)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('forecast_result/lp_vs_ppo_timeseries.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print('✓ Time series chart saved')


In [ ]:
# Distribution của số VM
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

datasets = [
    (lp_df, 'LP (Reactive)', colors['LP (Reactive)']),
    (ppo_cost_df, 'PPO (Cost)', colors['PPO (Cost)']),
    (ppo_overload_df, 'PPO (Overload)', colors['PPO (Overload)'])
]

for ax, (df, name, color) in zip(axes, datasets):
    vm_counts = df['vm_count'].value_counts().sort_index()
    ax.bar(vm_counts.index, vm_counts.values, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{name}\nPhân bố số VM mỗi step', fontweight='bold')
    ax.set_xlabel('Số VM')
    ax.set_ylabel('Số lần xuất hiện')
    
    # Annotate percentages for top values
    total = len(df)
    for idx, val in vm_counts.head(5).items():
        pct = val / total * 100
        ax.annotate(f'{pct:.1f}%', xy=(idx, val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('forecast_result/lp_vs_ppo_vm_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print('✓ VM distribution chart saved')


## 5. Phân tích Chi tiết


In [ ]:
# Phân tích chi tiết hiệu quả
print('='*70)
print('                    PHÂN TÍCH CHI TIẾT')
print('='*70)

# 1. So sánh VM usage
print('\n📊 1. TỔNG SỐ MÁY ẢO SỬ DỤNG')
print('-'*50)
print(f"  LP (Reactive):     {lp_metrics['Total VMs Used']:>10,} VMs")
print(f"  PPO (Cost):        {ppo_cost_metrics['Total VMs Used']:>10,} VMs")
print(f"  PPO (Overload):    {ppo_overload_metrics['Total VMs Used']:>10,} VMs")

# So sánh với LP
if lp_metrics['Total VMs Used'] > 0:
    ppo_cost_vm_diff = (ppo_cost_metrics['Total VMs Used'] - lp_metrics['Total VMs Used']) / lp_metrics['Total VMs Used'] * 100
    ppo_overload_vm_diff = (ppo_overload_metrics['Total VMs Used'] - lp_metrics['Total VMs Used']) / lp_metrics['Total VMs Used'] * 100
    print(f"\n  → PPO (Cost) dùng {'nhiều hơn' if ppo_cost_vm_diff > 0 else 'ít hơn'} LP: {abs(ppo_cost_vm_diff):.1f}%")
    print(f"  → PPO (Overload) dùng {'nhiều hơn' if ppo_overload_vm_diff > 0 else 'ít hơn'} LP: {abs(ppo_overload_vm_diff):.1f}%")


In [ ]:
# 2. So sánh utilization
print('\n📊 2. TỶ LỆ SỬ DỤNG TÀI NGUYÊN (khi có VM)')
print('-'*50)
print(f"{'Method':<20} {'CPU Util %':>12} {'Mem Util %':>12}")
print('-'*50)
print(f"{'LP (Reactive)':<20} {lp_metrics['Mean CPU Util (VM steps) %']:>12.2f} {lp_metrics['Mean Mem Util (VM steps) %']:>12.2f}")
print(f"{'PPO (Cost)':<20} {ppo_cost_metrics['Mean CPU Util (VM steps) %']:>12.2f} {ppo_cost_metrics['Mean Mem Util (VM steps) %']:>12.2f}")
print(f"{'PPO (Overload)':<20} {ppo_overload_metrics['Mean CPU Util (VM steps) %']:>12.2f} {ppo_overload_metrics['Mean Mem Util (VM steps) %']:>12.2f}")

print('\n  💡 Ghi chú:')
print('     - CPU Utilization cao → VM được sử dụng hiệu quả')
print('     - Utilization quá thấp → over-provisioning (lãng phí)')
print('     - Ngưỡng tốt: 60-80%')


In [ ]:
# 3. So sánh cost
print('\n📊 3. CHI PHÍ THUÊ VẬN HÀNH VM')
print('-'*50)
print(f"{'Method':<20} {'Total Cost ($)':>15} {'Avg/Step ($/h)':>15}")
print('-'*50)
print(f"{'LP (Reactive)':<20} {lp_metrics['Actual Total Cost ($)']:>15.4f} {lp_metrics['Avg Cost/Step ($/h)']:>15.6f}")
print(f"{'PPO (Cost)':<20} {ppo_cost_metrics['Actual Total Cost ($)']:>15.4f} {ppo_cost_metrics['Avg Cost/Step ($/h)']:>15.6f}")
print(f"{'PPO (Overload)':<20} {ppo_overload_metrics['Actual Total Cost ($)']:>15.4f} {ppo_overload_metrics['Avg Cost/Step ($/h)']:>15.6f}")

# So sánh với LP
if lp_metrics['Actual Total Cost ($)'] > 0:
    ppo_cost_diff = (ppo_cost_metrics['Actual Total Cost ($)'] - lp_metrics['Actual Total Cost ($)']) / lp_metrics['Actual Total Cost ($)'] * 100
    ppo_overload_diff = (ppo_overload_metrics['Actual Total Cost ($)'] - lp_metrics['Actual Total Cost ($)']) / lp_metrics['Actual Total Cost ($)'] * 100
    print(f"\n  → PPO (Cost) {'đắt hơn' if ppo_cost_diff > 0 else 'rẻ hơn'} LP: {abs(ppo_cost_diff):.1f}%")
    print(f"  → PPO (Overload) {'đắt hơn' if ppo_overload_diff > 0 else 'rẻ hơn'} LP: {abs(ppo_overload_diff):.1f}%")


In [ ]:
# SLA violations
print('\n📊 4. VI PHẠM SLA')
print('-'*50)
print(f"  LP (Reactive):     {lp_metrics['SLA Violations']:>6} violations ({lp_metrics['SLA Violation Rate %']:.2f}%)")
print(f"  PPO (Cost):        {ppo_cost_metrics['SLA Violations']:>6} violations ({ppo_cost_metrics['SLA Violation Rate %']:.2f}%)")
print(f"  PPO (Overload):    {ppo_overload_metrics['SLA Violations']:>6} violations ({ppo_overload_metrics['SLA Violation Rate %']:.2f}%)")


## 6. Kết luận & Nhận xét


In [ ]:
print('='*70)
print('                         KẾT LUẬN')
print('='*70)

# Xác định phương pháp tốt nhất cho từng tiêu chí
all_metrics = [lp_metrics, ppo_cost_metrics, ppo_overload_metrics]

# VM: ít nhất là tốt nhất
best_vm = min(all_metrics, key=lambda x: x['Total VMs Used'])
# Utilization: cao nhất là tốt nhất (hiệu quả)
best_util = max(all_metrics, key=lambda x: x['Mean CPU Util (VM steps) %'])
# Cost: thấp nhất là tốt nhất
best_cost = min(all_metrics, key=lambda x: x['Actual Total Cost ($)'])
# SLA: ít nhất là tốt nhất
best_sla = min(all_metrics, key=lambda x: x['SLA Violations'])

print(f"\n🏆 Tốt nhất về SỐ VM SỬ DỤNG:        {best_vm['Method']}")
print(f"🏆 Tốt nhất về HIỆU QUẢ TÀI NGUYÊN:  {best_util['Method']}")
print(f"🏆 Tốt nhất về CHI PHÍ VẬN HÀNH:     {best_cost['Method']}")
print(f"🏆 Tốt nhất về SLA COMPLIANCE:       {best_sla['Method']}")

print('\n' + '-'*70)
print('NHẬN XÉT:')
print('-'*70)
print("""
• LP (Linear Programming):
  - Giải tối ưu chính xác tại mỗi thời điểm
  - Không xem xét trạng thái tương lai (reactive)
  - Phù hợp khi cần đảm bảo tối ưu cục bộ

• PPO (Cost scenario):
  - Học từ dữ liệu, có thể dự đoán xu hướng
  - Ưu tiên tối ưu chi phí vận hành
  - Post-processing: Host-only khi không có overflow

• PPO (Overload scenario):
  - Ưu tiên giảm thiểu SLA violations
  - Có thể over-provision để đảm bảo capacity
""")


In [ ]:
# Lưu kết quả so sánh
comparison_result = {
    'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'methods': {
        'LP_Reactive': lp_metrics,
        'PPO_Cost': ppo_cost_metrics,
        'PPO_Overload': ppo_overload_metrics,
    },
    'best_for': {
        'vm_count': best_vm['Method'],
        'utilization': best_util['Method'],
        'cost': best_cost['Method'],
        'sla': best_sla['Method'],
    }
}

with open(RESULTS_DIR / 'lp_vs_ppo_comparison.json', 'w') as f:
    json.dump(comparison_result, f, indent=2)

print('\n✓ Comparison results saved to forecast_result/lp_vs_ppo_comparison.json')
print('✓ Charts saved to forecast_result/')
